# Content-Based Filtering with TF-IDF

**Portfolio notebook · Python, scikit-learn, Pandas, NumPy, and Matplotlib**

## Project summary

This notebook develops a content-based recommender from Amazon product-review text. It builds an item profile from review titles and text, represents profiles with TF-IDF vectors, measures item similarity with cosine similarity, and produces personalized recommendations from a user's rated-item history.

It complements the popularity baseline in the accompanying notebook. Popularity offers a dependable default for new users; content-based filtering adds personalization when descriptions and interaction history are available.

### What this notebook demonstrates

- Construction of item text profiles from review content
- TF-IDF feature engineering with unigrams and bigrams
- Item-to-item similarity and Top-N similar-item retrieval
- Rating-weighted user profiles and unseen-item recommendations
- Diversity analysis using intra-list similarity

## 1. Background Theory

### 1.1 The Content-Based Filtering Paradigm

Content-based filtering models the problem as follows. For each item $i$, we construct a **feature vector** $\mathbf{x}_i \in \mathbb{R}^d$ derived from its attributes. For a user $u$ with interaction history $H_u = \{i_1, i_2, \ldots\}$, we construct a **user profile vector** $\mathbf{p}_u$ that represents their aggregate preferences. Recommendations are then produced by finding items whose feature vectors are most similar to $\mathbf{p}_u$.

### 1.2 Term Frequency-Inverse Document Frequency (TF-IDF)

Raw word counts are a poor representation of text because common words ("the", "and", "is") dominate the vocabulary while contributing no discriminative signal. TF-IDF reweights word counts to amplify terms that are informative for a specific document and rare across the corpus.

For a term $t$ in document $d$ within corpus $D$:

**Term Frequency:**
$$\text{TF}(t, d) = \frac{\text{count of } t \text{ in } d}{\text{total terms in } d}$$

**Inverse Document Frequency:**
$$\text{IDF}(t, D) = \log\left(\frac{|D|}{1 + |\{d \in D : t \in d\}|}\right)$$

**TF-IDF weight:**
$$\text{TF-IDF}(t, d, D) = \text{TF}(t, d) \times \text{IDF}(t, D)$$

The resulting matrix $\mathbf{X} \in \mathbb{R}^{n \times V}$ has one row per item and one column per vocabulary term, with each cell holding the TF-IDF weight. This matrix is typically very sparse since each document uses only a small subset of the full vocabulary.

### 1.3 Cosine Similarity

Given two item vectors $\mathbf{x}_i$ and $\mathbf{x}_j$, their **cosine similarity** measures the angle between them in the high-dimensional TF-IDF space:

$$\text{sim}(i, j) = \cos(\theta) = \frac{\mathbf{x}_i \cdot \mathbf{x}_j}{\|\mathbf{x}_i\| \cdot \|\mathbf{x}_j\|}$$

Cosine similarity ranges from $-1$ to $1$, but since TF-IDF weights are non-negative it ranges from $0$ to $1$ in practice. It is preferred over Euclidean distance for text vectors because it is invariant to document length — a short and a long product description covering the same topics will still show high similarity.

For the full item set, we compute the similarity matrix $S \in \mathbb{R}^{n \times n}$ where $S_{ij} = \text{sim}(i, j)$. The diagonal entries are all 1 (an item is perfectly similar to itself).

### 1.4 User Profile Construction

A simple and effective user profile aggregates the TF-IDF vectors of items the user has interacted with, weighted by their ratings:

$$\mathbf{p}_u = \frac{\sum_{i \in H_u} r_{ui} \cdot \mathbf{x}_i}{\sum_{i \in H_u} r_{ui}}$$

This weighted average shifts the profile toward items the user rated highly and away from items they rated poorly. The profile can then be compared against all item vectors using cosine similarity to produce a ranked recommendation list.

### 1.5 Limitations of Content-Based Filtering

Content-based approaches have several well-documented failure modes:

- **Over-specialization (filter bubble):** The model recommends items very similar to what the user has already seen, limiting serendipity and diversity.
- **Feature quality dependence:** Recommendations are only as good as the item descriptions. Sparse or low-quality text yields poor item vectors.
- **No cross-user signal:** The model cannot discover that two users with different histories might enjoy the same novel item.
- **User cold-start:** A new user with no history has no profile vector and cannot receive personalized recommendations.

## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

## 2. Data loading and item-profile construction

The notebook expects a compatible Amazon Reviews JSON-lines dataset in the local `data/` folder. The dataset is intentionally excluded from version control because it can be large and may have separate distribution conditions.

In [ ]:
from pathlib import Path

DATA_PATH = Path("data/Electronics_5.json.gz")
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found: {DATA_PATH}. Add a compatible Amazon Reviews JSON-lines file "
        "to the local data/ folder before running this notebook."
    )

df = pd.read_json(DATA_PATH, lines=True, compression="infer")
df = df.rename(columns={"reviewerID": "user_id", "asin": "item_id", "overall": "rating"})
required_columns = ["user_id", "item_id", "rating", "reviewText", "summary"]
df = df.dropna(subset=["user_id", "item_id", "rating"])
df = df.reindex(columns=required_columns, fill_value="")
df["reviewText"] = df["reviewText"].fillna("").astype(str)
df["summary"] = df["summary"].fillna("").astype(str)

print(f"Loaded reviews: {len(df):,}")
df.head()

In [ ]:
# Each item receives one text profile made from the review text and summary fields.
items = (
    df.assign(combined_text=df["reviewText"] + " " + df["summary"])
      .groupby("item_id", as_index=False)["combined_text"]
      .agg(" ".join)
      .rename(columns={"combined_text": "text_profile"})
)

print(f"Items with text profiles: {len(items):,}")
items.head()

## 3. TF-IDF feature engineering

In [ ]:
vectorizer = TfidfVectorizer(
    max_features=10_000,
    stop_words="english",
    ngram_range=(1, 2),
    min_df=2,
)
tfidf_matrix = vectorizer.fit_transform(items["text_profile"])

print(f"TF-IDF matrix shape: {tfidf_matrix.shape}")
print(f"Matrix sparsity: {1 - tfidf_matrix.nnz / (tfidf_matrix.shape[0] * tfidf_matrix.shape[1]):.4%}")

In [ ]:
# Inspect the most important terms for a sample item
def top_terms_for_item(item_idx, vectorizer, tfidf_matrix, n=15):
    """Return the top-n TF-IDF terms for a given item index."""
    feature_names = vectorizer.get_feature_names_out()
    row = tfidf_matrix[item_idx]
    scores = zip(feature_names, np.asarray(row.todense()).flatten())
    sorted_scores = sorted(scores, key=lambda x: x[1], reverse=True)
    return sorted_scores[:n]

print(top_terms_for_item(0, vectorizer, tfidf_matrix))


## 4. Item-to-item similarity

A full pairwise similarity matrix grows quadratically with the number of items. To keep this notebook practical on a laptop, the analysis below limits the matrix to the most-reviewed products. Adjust `MAX_ITEMS` according to available memory.

In [ ]:
MAX_ITEMS = 5_000
item_counts = df["item_id"].value_counts().head(MAX_ITEMS)
items = items[items["item_id"].isin(item_counts.index)].reset_index(drop=True)

tfidf_matrix = vectorizer.fit_transform(items["text_profile"])
sim_matrix = cosine_similarity(tfidf_matrix)
item_to_idx = {item_id: idx for idx, item_id in enumerate(items["item_id"])}

print(f"Similarity matrix shape: {sim_matrix.shape}")

In [ ]:
def get_similar_items(item_id, sim_matrix, item_to_idx, items_df, n=10):
    """Return the top-N most similar items, excluding the query item itself."""
    if item_id not in item_to_idx:
        raise KeyError(f"Unknown item_id: {item_id}")

    idx = item_to_idx[item_id]
    scores = sim_matrix[idx]
    candidate_indices = np.argsort(scores)[::-1]
    candidate_indices = candidate_indices[candidate_indices != idx][:n]

    result = items_df.iloc[candidate_indices][["item_id"]].copy()
    result["similarity"] = scores[candidate_indices]
    return result.sort_values("similarity", ascending=False)

sample_item_id = items["item_id"].iloc[0]
get_similar_items(sample_item_id, sim_matrix, item_to_idx, items, n=10)

## 5. Personalized recommendations from user profiles

A user profile is formed by the rating-weighted average of the TF-IDF item vectors in that user's history. Candidate items already rated by the user are removed before ranking.

In [ ]:
def build_user_profile(user_id, df, item_to_idx, tfidf_matrix):
    """Build a rating-weighted TF-IDF profile or return None when history is unavailable."""
    user_df = df[df["user_id"] == user_id]
    user_df = user_df[user_df["item_id"].isin(item_to_idx)]
    if user_df.empty:
        return None

    item_indices = user_df["item_id"].map(item_to_idx).to_numpy()
    ratings = user_df["rating"].astype(float).to_numpy()
    if ratings.sum() <= 0:
        return None

    weighted_vectors = tfidf_matrix[item_indices].multiply(ratings.reshape(-1, 1))
    return np.asarray(weighted_vectors.sum(axis=0)) / ratings.sum()


def recommend_for_user(user_id, df, item_to_idx, tfidf_matrix, items_df, n=10):
    """Return Top-N unseen items ranked by cosine similarity to the user profile."""
    profile = build_user_profile(user_id, df, item_to_idx, tfidf_matrix)
    if profile is None:
        return pd.DataFrame(columns=["item_id", "similarity"])

    scores = cosine_similarity(profile, tfidf_matrix).flatten()
    seen = set(df.loc[df["user_id"] == user_id, "item_id"])
    candidate_indices = [i for i, item_id in enumerate(items_df["item_id"]) if item_id not in seen]
    ranked = sorted(candidate_indices, key=lambda i: scores[i], reverse=True)[:n]

    result = items_df.iloc[ranked][["item_id"]].copy()
    result["similarity"] = scores[ranked]
    return result.sort_values("similarity", ascending=False)

eligible_users = df["user_id"].value_counts()
sample_user_id = eligible_users[eligible_users >= 10].index[0]
recommend_for_user(sample_user_id, df, item_to_idx, tfidf_matrix, items, n=10)

## 6. Diversity and filter-bubble analysis

Content-based recommenders can over-specialize by repeatedly presenting items that resemble the user's past interactions. Intra-list similarity (ILS) measures the average pairwise similarity of items in a recommendation list: a lower score indicates a more diverse list.

In [ ]:
def intra_list_similarity(recommended_item_ids, sim_matrix, item_to_idx):
    """Return mean pairwise similarity for a recommendation list."""
    indices = [item_to_idx[item_id] for item_id in recommended_item_ids if item_id in item_to_idx]
    if len(indices) < 2:
        return np.nan

    submatrix = sim_matrix[np.ix_(indices, indices)]
    return (submatrix.sum() - np.trace(submatrix)) / (len(indices) * (len(indices) - 1))

recommendations = recommend_for_user(sample_user_id, df, item_to_idx, tfidf_matrix, items, n=10)
ils_score = intra_list_similarity(recommendations["item_id"].tolist(), sim_matrix, item_to_idx)
print(f"Intra-list similarity: {ils_score:.4f}")

## 7. Interpretation and next steps

TF-IDF with unigrams and bigrams captures meaningful terms and short phrases from item text, which makes the method useful when rich descriptions are available. However, the richer vocabulary also increases sparsity and can make weak or repetitive descriptions less useful.

Content-based filtering is particularly valuable for item cold-start because a new product can be represented immediately from its text. Its main limitation is over-specialization: it cannot learn from similar users or easily surface unexpected items. A practical next step would be a hybrid recommender that combines this content score with the Bayesian-popularity baseline and collaborative interaction signals.